# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tu-h-nguyn/FlyRank-End-to-End-Machine-Learning-Project-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

Lane 2 — Refresh / Content Opportunity Scoring. Notebook này biến điểm số của mô hình thành thứ mà một
biên tập viên thực sự dùng được: **một hàng đợi có thứ tự, mỗi dòng kèm lý do, hành động đề xuất và mức
tin cậy** — cộng với giới hạn sử dụng, danh sách không-bao-giờ-tự-động, và điều kiện huấn luyện lại.

## 1. Ranked actions + reason codes

Điểm rủi ro đến từ mô hình hồi quy logistic, chấm **out-of-fold** (mỗi trang được chấm bởi một mô hình
chưa từng thấy client của nó). Nhưng điểm số trần trụi thì không ai tin. Vì vậy mỗi dòng mang thêm:

| Thành phần | Ý nghĩa |
|---|---|
| `reason_codes` | Vì sao trang này ở đây — đọc trực tiếp từ dữ liệu, không phải từ mô hình |
| `action` | Việc cụ thể đề xuất làm |
| `confidence` | high / medium / low, kết hợp điểm rủi ro với khối lượng nhu cầu |

**Bảng ánh xạ reason code → hành động** (chính sách do tôi chọn, không phải chân lý):

| Reason code | Nghĩa (quan sát được) | Hành động |
|---|---|---|
| `visibility_slipping` | Impressions giảm > 20% giữa hai cửa sổ 30 ngày đã nhìn thấy được | `review_for_refresh` |
| `visibility_slipping` + `high_demand_page` | Đang tụt, nhưng vẫn còn ≥ 3,000 impressions/30 ngày | `protect_and_refresh` (ưu tiên cao nhất) |
| `clicks_falling_while_visible` | Hiển thị giữ nguyên nhưng click rơi > 20% | `review_metadata_and_intent` |
| `low_ctr_high_exposure` | CTR dưới trung vị dù có ≥ 500 impressions | `review_metadata_and_intent` |
| `aging_content` | Nội dung ≥ 365 ngày tuổi tại thời điểm ra quyết định | `review_for_update` |
| `valuable_keyword` | Từ khóa mục tiêu có search volume ≥ 1,000 | (nâng mức ưu tiên) |
| `growing_with_demand` | Impressions cửa sổ trước tăng > 20% và còn ≥ 500 impressions | `protect_and_watch` |
| `spiking_may_revert` | Impressions cửa sổ trước bùng > 50% — bằng chứng ở `w04_signal_audit`: nhóm này suy giảm 58.7%, cao hơn đáy 52.4% | `protect_and_watch` |
| `model_pattern_only` | Mô hình thấy rủi ro nhưng không quy tắc đơn giản nào khớp | `monitor` — luôn cần người đọc trước |

`model_pattern_only` tồn tại có chủ đích: khi mô hình không giải thích được bằng ngôn ngữ con người,
nó phải **tự hạ mức đề xuất của mình** xuống mức theo dõi, chứ không phải im lặng đẩy trang lên đầu.

### Bốn trạng thái mà lane yêu cầu — và ranh giới giữa chúng

Card yêu cầu chấm điểm trang đang **growing / declining / recovering / worth review**. Bốn trạng thái
này **không nằm cùng một phía** của thời điểm ra quyết định, nên hàng đợi tách hẳn thành hai cột:

| Cột | Tính từ đâu | Dùng được để làm gì |
|---|---|---|
| `decision_state` | **Chỉ cửa sổ trước** — `slipping` / `steady` / `spiking` | **Hành động được.** Biên tập viên thấy được ngay tại thời điểm quyết định |
| `outcome_state` | Cửa sổ kết quả — `declining` / `recovering` / `growing` / `stable` | **Chỉ để báo cáo và đánh giá.** Đây là thông tin phía nhãn: không bao giờ làm feature, và engine không hề dự báo nó |

**Vì sao `recovering` báo cáo được nhưng không dự báo được:** phát hiện hồi phục *trước khi nó xảy ra*
cần **hai** mức thay đổi liên tiếp (tụt rồi bật lại). Slice này chỉ cho thấy **hai** cửa sổ trước
quyết định, tức là **một** mức thay đổi. Nên engine nói được "1,063 trang đã hồi phục", nhưng không nói
trước được trang nào sẽ hồi phục. Muốn có thì cần bảng daily của warehouse, không phải mô hình tốt hơn.

In [1]:
# --- Bootstrap: chạy được cả ở local lẫn trên Colab ---
import os, sys, urllib.request

BRANCHES = [
    "https://raw.githubusercontent.com/tu-h-nguyn/FlyRank-End-to-End-Machine-Learning-Project-Internship/main",
    "https://raw.githubusercontent.com/tu-h-nguyn/FlyRank-End-to-End-Machine-Learning-Project-Internship/claude/search-ranking-capstone-166z1j",
]

def _pipeline_dir() -> str:
    """Toàn bộ logic capstone nằm trong MỘT file: work/scripts/capstone_pipeline.py."""
    for p in ["../scripts", "work/scripts", "scripts", "../../work/scripts"]:
        if os.path.exists(os.path.join(p, "capstone_pipeline.py")):
            return p
    os.makedirs("work/scripts", exist_ok=True)
    for base in BRANCHES:
        try:
            urllib.request.urlretrieve(f"{base}/work/scripts/capstone_pipeline.py",
                                       "work/scripts/capstone_pipeline.py")
            return "work/scripts"
        except Exception:
            continue
    raise RuntimeError("Không tải được capstone_pipeline.py")

sys.path.insert(0, _pipeline_dir())
import numpy as np
import pandas as pd
import capstone_pipeline as cp

raw = cp.load_raw()
frame = cp.build_frame(raw)
d, population = cp.apply_population_filter(frame)
y = d["label_declined"].to_numpy()
groups = d["client_id"].to_numpy()
X = cp.design_matrix(d)

print(f"Population: {population['rows_modelled']:,} trang / {population['clients_modelled']} client "
      f"(lọc từ {population['rows_start']:,} dòng, ngưỡng impressions_prev_30d >= {cp.MIN_PREV_IMPRESSIONS})")
print(f"Base rate (tỷ lệ trang thực sự suy giảm > 20% trong 30 ngày kế tiếp): {y.mean():.4f}")

from sklearn.metrics import average_precision_score

# Điểm out-of-fold, split nhóm theo client — giống hệt con số trong paper
oof = cp.oof_scores(X, y, groups, cp.PRIMARY_MODEL)

q = d.copy()
q["risk_score"] = oof
ctr_median = float(q["prior_ctr"].median())
pairs = q.apply(lambda r: cp.reason_codes(r, ctr_median), axis=1)
q["reason_codes"] = [c for c, _ in pairs]
q["action"] = [a for _, a in pairs]
q["confidence"] = [cp.confidence_label(p, i)
                   for p, i in zip(q["risk_score"], q["impressions_prev_30d"])]
q["decision_state"] = cp.decision_state(q)   # hành động được
q["outcome_state"] = cp.outcome_state(q)     # chỉ để báo cáo — không bao giờ là input
q = q.sort_values("risk_score", ascending=False).reset_index(drop=True)

show = ["content_id", "risk_score", "action", "decision_state", "reason_codes", "confidence",
        "impressions_prev_30d", "prior_impr_trend_pct", "outcome_state", "label_declined"]
print("TOP 20 CỦA HÀNG ĐỢI (content_id là mã ẩn danh — an toàn để công bố)")
display(q.head(20)[show].round({"risk_score": 3, "prior_impr_trend_pct": 1, "prior_ctr": 2}))

base = float(y.mean())
for k in [20, 50, 100, 200, 500]:
    p = cp.precision_at_k(q["risk_score"].to_numpy(), q["label_declined"].to_numpy(), k)
    print(f"Precision@{k:<4} = {p:.3f}  (base rate {base:.3f}, lift {p/base:.2f}x)")

print("\nPhân bổ hành động trong 200 trang đầu:")
print(q.head(200)["action"].value_counts().to_string())
print("\nPhân bổ mức tin cậy trong 200 trang đầu:")
print(q.head(200)["confidence"].value_counts().to_string())
print("\nPhân bổ hành động trên toàn bộ hàng đợi (18,010 trang):")
print(q["action"].value_counts().to_string())

print("\n--- BỐN TRẠNG THÁI ---")
print("decision_state (hành động được, chỉ dùng cửa sổ trước) — top 200:")
print(q.head(200)["decision_state"].value_counts().to_string())
print("\noutcome_state (CHỈ để báo cáo — chuyện đã xảy ra ở cửa sổ kết quả):")
for k in [50, 200, 500, 1000]:
    vc = q.head(k)["outcome_state"].value_counts(normalize=True)
    print(f"  top {k:>4}: " + "  ".join(f"{st} {vc.get(st, 0):.0%}"
          for st in ["declining", "recovering", "growing", "stable"]))
vc = q["outcome_state"].value_counts(normalize=True)
print("  toàn bộ : " + "  ".join(f"{st} {vc.get(st, 0):.0%}"
      for st in ["declining", "recovering", "growing", "stable"]))
print("\n=> Đọc dòng top-50 cạnh dòng toàn bộ: hàng đợi dồn suy giảm lên đầu "
      "và đẩy trang đang tăng trưởng xuống dưới.")

Population: 18,010 trang / 30 client (lọc từ 30,000 dòng, ngưỡng impressions_prev_30d >= 100)
Base rate (tỷ lệ trang thực sự suy giảm > 20% trong 30 ngày kế tiếp): 0.6155


TOP 20 CỦA HÀNG ĐỢI (content_id là mã ẩn danh — an toàn để công bố)


,content_id,risk_score,action,decision_state,reason_codes,confidence,impressions_prev_30d,prior_impr_trend_pct,outcome_state,label_declined
0,content_d4786a3f7544,0.963,review_for_refresh,slipping,visibility_slipping|low_ctr_high_exposure,high,2775,-22.7,declining,1
1,content_d8fe64aaac6b,0.956,review_for_refresh,slipping,visibility_slipping|low_ctr_high_exposure,medium,850,-48.5,declining,1
2,content_ec5e5f49929b,0.949,review_for_refresh,slipping,visibility_slipping|low_ctr_high_exposure,high,1952,-52.9,declining,1
3,content_7760048e1609,0.948,review_metadata_and_intent,steady,low_ctr_high_exposure,high,1597,-8.5,declining,1
4,content_900f828b925b,0.947,review_metadata_and_intent,spiking,low_ctr_high_exposure|growing_with_demand,medium,927,25.6,declining,1
5,content_82f523ddc57e,0.945,review_for_refresh,slipping,visibility_slipping|low_ctr_high_exposure,high,2824,-43.9,declining,1
6,content_e9cc09b619e7,0.943,review_metadata_and_intent,steady,clicks_falling_while_visible|low_ctr_high_expo...,high,1341,-15.4,declining,1
7,content_b23fa9e12c1d,0.942,review_for_refresh,slipping,visibility_slipping,medium,143,-84.3,declining,1
8,content_b8545cf96378,0.940,review_for_refresh,slipping,visibility_slipping|low_ctr_high_exposure,medium,635,-35.9,declining,1
9,content_aa6cb9d42da5,0.939,review_for_refresh,slipping,visibility_slipping|low_ctr_high_exposure,medium,667,-24.3,declining,1


Precision@20   = 0.900  (base rate 0.616, lift 1.46x)
Precision@50   = 0.880  (base rate 0.616, lift 1.43x)
Precision@100  = 0.830  (base rate 0.616, lift 1.35x)
Precision@200  = 0.815  (base rate 0.616, lift 1.32x)
Precision@500  = 0.784  (base rate 0.616, lift 1.27x)

Phân bổ hành động trong 200 trang đầu:
action
review_for_refresh            97
review_metadata_and_intent    62
protect_and_refresh           27
monitor                        9
protect_and_watch              5

Phân bổ mức tin cậy trong 200 trang đầu:
confidence
high      137
medium     63

Phân bổ hành động trên toàn bộ hàng đợi (18,010 trang):
action
review_for_refresh            7943
review_metadata_and_intent    3352
protect_and_watch             2344
monitor                       2218
protect_and_refresh           1273
review_for_update              880

--- BỐN TRẠNG THÁI ---
decision_state (hành động được, chỉ dùng cửa sổ trước) — top 200:
decision_state
slipping    124
steady       41
spiking      35

outcome_s

## 2. Intended use and limits

**Ai dùng:** biên tập viên nội dung / chuyên viên SEO, đầu mỗi chu kỳ rà soát.

**Dùng để làm gì:** quyết định **thứ tự** rà soát khi năng lực có hạn (ví dụ 50 trang/tuần).
Đây là danh sách xếp hạng ứng viên, không phải danh sách việc phải làm.

**Nó KHÔNG dùng để:**

- Hứa hẹn rằng refresh sẽ khôi phục lưu lượng — dữ liệu này không có thiết kế nhân quả nào cả.
- Đánh giá chất lượng bài viết hay hiệu suất của người viết.
- Tự động chỉnh sửa, xuất bản, gỡ hay redirect bất cứ trang nào.
- Suy ra bất cứ điều gì về thuật toán của Google.

**Nó hết hiệu lực ở đâu (validity boundary):**

- **Ngoài dân số đã mô hình hóa:** chỉ áp dụng cho trang có ≥ 100 impressions ở cửa sổ 30 ngày trước.
  11,990/30,000 trang volume thấp bị loại — với nhóm đó, danh sách này im lặng chứ không phải nói "an toàn".
- **Chỉ 30 client ẩn danh** trong một snapshot. PR-AUC dao động 0.63–0.70 khi đổi tập client giữ lại.
- **Ngưỡng "down" = giảm > 20%** là lựa chọn chính sách kế thừa từ định nghĩa sản phẩm, không phải hằng số.
- **Tính mùa vụ chưa được kiểm soát:** một snapshot 90 ngày không tách được suy giảm thật khỏi chu kỳ mùa vụ.

In [2]:
model_card = {
    "name": "Refresh Opportunity Ranker (Lane 2)",
    "version": "capstone-1.0",
    "what_it_outputs": "Điểm rủi ro 0-1 + reason codes + hành động đề xuất + mức tin cậy",
    "decision_it_supports": "Trang nào biên tập viên nên rà soát TRƯỚC trong tuần này",
    "unit_of_analysis": "một trang nội dung ẩn danh (content_id) trong một snapshot",
    "label": "impressions 30 ngày gần nhất giảm > 20% so với 30 ngày liền trước",
    "decision_point": "đầu cửa sổ 30 ngày gần nhất",
    "training_population": f"{len(q):,} trang / {d['client_id'].nunique()} client, "
                           f"impressions_prev_30d >= {cp.MIN_PREV_IMPRESSIONS}",
    "validation": f"GroupKFold({cp.N_SPLITS}) theo client_id, mọi chỉ số out-of-fold",
    "headline": {
        "base_rate": round(base, 4),
        "precision_at_50": cp.precision_at_k(q['risk_score'].to_numpy(),
                                             q['label_declined'].to_numpy(), 50),
        "pr_auc": round(float(average_precision_score(y, oof)), 4),
    },
    "known_failure_modes": [
        "Trang volume thấp (< 100 impressions/30 ngày) nằm ngoài phạm vi",
        "Không phân biệt được suy giảm thật với tính mùa vụ hoặc consolidation",
        "Không biết gì về nội dung thật của trang (không đọc text, title, URL)",
        "Không dùng được days_since_last_update: 68% trang được cập nhật trong cửa sổ kết quả",
    ],
    "not_for": ["tự động xuất bản", "đánh giá nhân sự", "claim nhân quả", "claim về thuật toán Google"],
}
import json as _json
print(_json.dumps(model_card, indent=2, ensure_ascii=False))

{
  "name": "Refresh Opportunity Ranker (Lane 2)",
  "version": "capstone-1.0",
  "what_it_outputs": "Điểm rủi ro 0-1 + reason codes + hành động đề xuất + mức tin cậy",
  "decision_it_supports": "Trang nào biên tập viên nên rà soát TRƯỚC trong tuần này",
  "unit_of_analysis": "một trang nội dung ẩn danh (content_id) trong một snapshot",
  "label": "impressions 30 ngày gần nhất giảm > 20% so với 30 ngày liền trước",
  "decision_point": "đầu cửa sổ 30 ngày gần nhất",
  "training_population": "18,010 trang / 30 client, impressions_prev_30d >= 100",
  "validation": "GroupKFold(5) theo client_id, mọi chỉ số out-of-fold",
  "headline": {
    "base_rate": 0.6155,
    "precision_at_50": 0.88,
    "pr_auc": 0.7183
  },
  "known_failure_modes": [
    "Trang volume thấp (< 100 impressions/30 ngày) nằm ngoài phạm vi",
    "Không phân biệt được suy giảm thật với tính mùa vụ hoặc consolidation",
    "Không biết gì về nội dung thật của trang (không đọc text, title, URL)",
    "Không dùng được days_si

## 3. Human review + the no-go list

**Con người phải kiểm tra gì trước khi hành động (mỗi trang, mất khoảng 2 phút):**

1. **Trang có còn đúng chủ đề kinh doanh không?** Impressions cao ở từ khóa vô giá trị vẫn ra điểm cao.
2. **Đây có phải consolidation không?** Một trang anh em trên cùng site có hút mất nhu cầu không?
   Mô hình chỉ nhìn từng trang riêng lẻ, nó không thể thấy điều này.
3. **Có phải mùa vụ không?** Nếu chủ đề tự nhiên rơi theo lịch, "suy giảm" là bình thường.
4. **Nội dung có thực sự lỗi thời không?** Trang evergreen (định nghĩa, khái niệm nền) có thể đúng mãi mãi.
5. **Vị trí có nằm trong tầm với không?** Trang ở vị trí 60–80 hiếm khi lên trang 1 chỉ bằng một lần refresh.
6. **`confidence = low` hoặc `model_pattern_only`?** Bắt buộc người đọc, không được xử lý hàng loạt.

**Danh sách KHÔNG BAO GIỜ tự động hóa:**

- Xuất bản, viết lại, gộp, xóa, hay redirect bất kỳ trang nào dựa trên điểm số.
- Gửi điểm số cho khách hàng như thể đó là dự báo doanh thu.
- Thay đổi title / meta description hàng loạt bằng script.
- Dùng điểm số này để đánh giá con người.
- Đưa `risk_score` ngược trở lại làm feature cho một mô hình sau — đó là vòng lặp tự khẳng định
  (chính là cái bẫy "product flag as feature" mà toàn bộ dự án này tránh).

In [3]:
review_gates = pd.DataFrame([
    {"Cổng kiểm tra": "Giá trị kinh doanh của từ khóa", "Ai làm": "Biên tập viên",
     "Chặn hành động nếu": "Impressions cao nhưng chủ đề không liên quan mục tiêu"},
    {"Cổng kiểm tra": "Kiểm tra consolidation", "Ai làm": "SEO",
     "Chặn hành động nếu": "Một trang anh em đã hút nhu cầu — nên gộp, không nên refresh"},
    {"Cổng kiểm tra": "Kiểm tra mùa vụ", "Ai làm": "SEO",
     "Chặn hành động nếu": "Chủ đề rơi theo lịch hằng năm"},
    {"Cổng kiểm tra": "Nội dung evergreen", "Ai làm": "Biên tập viên",
     "Chặn hành động nếu": "Nội dung vẫn đúng, không có gì để cập nhật"},
    {"Cổng kiểm tra": "Vị trí trong tầm với", "Ai làm": "SEO",
     "Chặn hành động nếu": "Vị trí trung bình quá sâu để một lần refresh thay đổi được"},
    {"Cổng kiểm tra": "Mức tin cậy", "Ai làm": "Hệ thống",
     "Chặn hành động nếu": "confidence = low hoặc reason = model_pattern_only"},
])
display(review_gates)

low_conf = int((q.head(200)["confidence"] == "low").sum())
pattern_only = int(q.head(200)["reason_codes"].str.contains("model_pattern_only").sum())
print(f"Trong 200 trang đầu: {low_conf} trang confidence=low, "
      f"{pattern_only} trang chỉ có model_pattern_only => bắt buộc người đọc trước khi làm gì.")

,Cổng kiểm tra,Ai làm,Chặn hành động nếu
0,Giá trị kinh doanh của từ khóa,Biên tập viên,Impressions cao nhưng chủ đề không liên quan m...
1,Kiểm tra consolidation,SEO,"Một trang anh em đã hút nhu cầu — nên gộp, khô..."
2,Kiểm tra mùa vụ,SEO,Chủ đề rơi theo lịch hằng năm
3,Nội dung evergreen,Biên tập viên,"Nội dung vẫn đúng, không có gì để cập nhật"
4,Vị trí trong tầm với,SEO,Vị trí trung bình quá sâu để một lần refresh t...
5,Mức tin cậy,Hệ thống,confidence = low hoặc reason = model_pattern_only


Trong 200 trang đầu: 0 trang confidence=low, 9 trang chỉ có model_pattern_only => bắt buộc người đọc trước khi làm gì.


## 4. Monitoring / retrain triggers

Một hàng đợi không hỏng bằng một tiếng nổ — nó hỏng âm thầm. Bốn tín hiệu tôi sẽ theo dõi, mỗi tín hiệu
có một ngưỡng viết sẵn để không phải tranh luận lúc đang hoảng:

| Tín hiệu | Đo thế nào | Ngưỡng báo động | Việc phải làm |
|---|---|---|---|
| **Trôi base rate** | Tỷ lệ trang suy giảm ở kỳ mới | Lệch > 10 điểm % so với 61.6% | Kiểm tra thị trường/mùa vụ trước, rồi mới huấn luyện lại |
| **Precision@50 tụt** | Đối chiếu hàng đợi kỳ trước với kết quả thật | < 0.74 (bằng rule baseline) | Mô hình không còn đáng dùng — quay lại rule cho tới khi sửa xong |
| **Trôi phân phối feature** | Trung vị `log_impr_prev30`, `prior_ctr` | Lệch > 20% so với lúc huấn luyện | Kiểm tra pipeline đo lường trước khi đổ lỗi cho mô hình |
| **Danh mục đổi hình** | Số client / số trang đủ điều kiện | Thay đổi > 25% | Huấn luyện lại: dân số đã khác |

**Nhịp huấn luyện lại mặc định:** mỗi tháng, hoặc ngay khi một ngưỡng trên bị chạm.
Mỗi lần huấn luyện lại phải chạy lại toàn bộ leakage checklist trong `w06_validation_audit.ipynb` —
rò rỉ thường lẻn vào lúc thêm cột mới, chứ không phải lúc viết mô hình.

In [4]:
monitoring = {
    "baseline_recorded_at": "capstone run, seed 42",
    "base_rate": round(base, 4),
    "base_rate_alert_if_outside": [round(base - 0.10, 4), round(base + 0.10, 4)],
    "precision_at_50": cp.precision_at_k(q["risk_score"].to_numpy(),
                                         q["label_declined"].to_numpy(), 50),
    "precision_at_50_floor_rule_baseline": 0.74,
    "feature_medians": {
        "log_impr_prev30": round(float(d["log_impr_prev30"].median()), 4),
        "prior_ctr": round(float(d["prior_ctr"].median()), 4),
        "prior_impr_trend_pct": round(float(d["prior_impr_trend_pct"].median()), 4),
    },
    "feature_drift_alert_pct": 20,
    "population": {"rows": int(len(q)), "clients": int(d["client_id"].nunique())},
    "population_change_alert_pct": 25,
    "retrain_cadence": "hàng tháng, hoặc ngay khi một ngưỡng bị chạm",
    "on_retrain": "chạy lại toàn bộ leakage checklist của w06 trước khi tin bất kỳ con số nào",
}
out = os.path.join(cp.OUT)
os.makedirs(out, exist_ok=True)
with open(os.path.join(out, "monitoring_thresholds.json"), "w") as f:
    _json.dump(monitoring, f, indent=2, ensure_ascii=False)
print(_json.dumps(monitoring, indent=2, ensure_ascii=False))
print(f"\nĐã ghi: {os.path.join(out, 'monitoring_thresholds.json')}")

{
  "baseline_recorded_at": "capstone run, seed 42",
  "base_rate": 0.6155,
  "base_rate_alert_if_outside": [
    0.5155,
    0.7155
  ],
  "precision_at_50": 0.88,
  "precision_at_50_floor_rule_baseline": 0.74,
  "feature_medians": {
    "log_impr_prev30": 6.6821,
    "prior_ctr": 0.1137,
    "prior_impr_trend_pct": -21.248
  },
  "feature_drift_alert_pct": 20,
  "population": {
    "rows": 18010,
    "clients": 30
  },
  "population_change_alert_pct": 25,
  "retrain_cadence": "hàng tháng, hoặc ngay khi một ngưỡng bị chạm",
  "on_retrain": "chạy lại toàn bộ leakage checklist của w06 trước khi tin bất kỳ con số nào"
}

Đã ghi: /home/user/flyrank-ML-internship-starter/work/outputs/monitoring_thresholds.json


## 5. Exports for the paper

Chạy lại toàn bộ pipeline một lần nữa, từ đầu, để mọi con số trong paper truy ngược được về một file.
Đây là "receipts": nếu người đọc muốn kiểm tra, họ mở JSON, không cần tin lời tôi.

- `work/outputs/capstone_metrics.json` — base rate, mọi chỉ số của mọi mô hình, thiết kế split
- `work/outputs/capstone_importance.json` — permutation importance (out-of-fold)
- `work/outputs/capstone_coefficients.json` — hệ số chuẩn hóa, để đọc *hướng* của từng tín hiệu
- `work/outputs/capstone_queue_top20.json` — 20 dòng đầu của hàng đợi (mã ẩn danh)
- `work/outputs/capstone_queue_summary.json` — phân bổ reason code / hành động / mức tin cậy
- `work/outputs/monitoring_thresholds.json` — ngưỡng giám sát ở mục 4
- `work/figures/*.svg` — 5 biểu đồ paper nhúng trực tiếp

Riêng `work/outputs/refresh_action_queue.csv` (18,010 dòng) **cố tình không commit** — `.gitignore`
của repo chặn mọi dataset CSV. Nó được tái tạo bằng một lệnh.

In [5]:
metrics = cp.main()   # chạy lại toàn bộ: metrics + importance + queue + 5 biểu đồ

print("\n--- FILE ĐÃ SINH ---")
for folder in [cp.OUT, cp.FIG]:
    for f in sorted(os.listdir(folder)):
        size = os.path.getsize(os.path.join(folder, f))
        print(f"  {folder.name}/{f:<34} {size:>9,} bytes")

rows=18,010  clients=30  base_rate=0.6155


{
  "Week-4 rule (window-contaminated)": {
    "pr_auc": 0.6159,
    "p@50": 0.72,
    "p@200": 0.615,
    "roc": 0.5004
  },
  "Rule baseline (decision-time columns)": {
    "pr_auc": 0.6676,
    "p@50": 0.74,
    "p@200": 0.73,
    "roc": 0.5621
  },
  "Random ordering": {
    "pr_auc": 0.6169,
    "p@50": 0.58,
    "p@200": 0.635,
    "roc": 0.498
  },
  "Model (logistic regression, honest features)": {
    "pr_auc": 0.7183,
    "p@50": 0.88,
    "p@200": 0.815,
    "roc": 0.6406
  },
  "Model (random forest, honest features)": {
    "pr_auc": 0.7108,
    "p@50": 0.8,
    "p@200": 0.79,
    "roc": 0.6366
  },
  "Model (random forest, window-overlapping features)": {
    "pr_auc": 0.7749,
    "p@50": 1.0,
    "p@200": 0.995,
    "roc": 0.6865
  }
}
split gap: {'grouped_by_client_pr_auc': 0.7183, 'random_row_split_pr_auc': 0.7604, 'random_row_split_base_rate': 0.6185, 'gap': 0.0421}
stability: {'seeds': [42, 7, 2024, 101, 777], 'pr_auc_mean': 0.6678, 'pr_auc_std': 0.0258, 'pr_auc_min'

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.